# <font color="#418FDE" size="6.5" uppercase>**Transfer mit PyTorch**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Setzen Bildaugmentation, Dropout, Batch Normalization, Scheduler und Early Stopping kontrolliert ein. 
- Nutzen ein leichtes vortrainiertes torchvision-Modell als eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Modelle hinsichtlich Genauigkeit, Laufzeit, Größe und Offline-Fallback. 


## **1. Regularisierung in CNNs**

### **1.1. Gezielte Bildaugmentation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_01.jpg?v=1784815830" width="250">



>* Augmentation fördert robuste visuelle Merkmale
>* Realistische Varianten verbessern Generalisierung ohne neue Daten

>* Augmentation muss zur Aufgabe passen
>* Nur realistische, klassenlogische Veränderungen nutzen

>* Augmentation nur im Training einsetzen
>* Schrittweise testen und Validierung vergleichen



In [ ]:
#@title Python-Code - Gezielte Bildaugmentation

# Dieses Beispiel zeigt gezielte Bildaugmentation.
# Transformationen bleiben klein und fachlich plausibel.
# Die Grafik vergleicht Original und Varianten.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageEnhance

# Wir erzeugen ein kleines synthetisches RGB-Bild.
image_size = 96
image = np.full((image_size, image_size, 3), 235, dtype=np.uint8)

# Ein einfaches Objekt macht Veränderungen gut sichtbar.
image[28:70, 34:62] = [80, 140, 220]
image[18:34, 42:54] = [230, 180, 80]
image[62:78, 24:72] = [90, 180, 110]

# Diese Augmentationen sind bewusst moderat gewählt.
pil_image = Image.fromarray(image)
flipped = pil_image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
rotated = pil_image.rotate(12, resample=Image.Resampling.BILINEAR)
bright = ImageEnhance.Brightness(pil_image).enhance(0.75)

# Wir kombinieren vier Bilder in einer einzigen Anzeige.
variants = [pil_image, flipped, rotated, bright]
labels = ["Original", "Spiegelung", "Rotation 12°", "Dunkler"]
canvas = np.full((128, 512, 3), 255, dtype=np.uint8)

# Jedes Bild wird mit Abstand nebeneinander platziert.
for index, variant in enumerate(variants):
    x_start = 16 + index * 124
    canvas[16:112, x_start:x_start + 96] = np.array(variant)

print("Gezielte Augmentation verändert Trainingsbilder, nicht die Klassenidee.")
print("Beispiel: Spiegelung, kleine Rotation und Helligkeit bleiben plausibel.")
print("Zu starke oder fachlich falsche Transformationen können schaden.")

# Eine Achse reicht für den visuellen Vergleich.
fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(canvas)
ax.set_title("Kontrollierte Bildaugmentation an einem synthetischen Objekt")

# Die Beschriftungen erklären die vier Varianten.
ax.set_xticks([64, 188, 312, 436])
ax.set_xticklabels(labels)
ax.set_yticks([])
ax.set_xlabel("Bildvariante")
ax.set_ylabel("Pixelposition")
plt.show()



### **1.2. Dropout und BatchNorm**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_02.jpg?v=1784815828" width="250">



>* Dropout fördert robuste Merkmale statt Auswendiglernen
>* Rate und Trainingsmodus sorgfältig einstellen

>* BatchNorm stabilisiert Aktivierungen und beschleunigt Training
>* Wirkt leicht regularisierend, ersetzt Dropout nicht

>* Dropout und BatchNorm gezielt platzieren
>* Validierung zeigt echte Generalisierung



In [ ]:
#@title Python-Code - Dropout und BatchNorm

# Dieses Beispiel zeigt Dropout und BatchNorm praktisch.
# Wir vergleichen Trainingsmodus und Auswertungsmodus.
# Die Ausgabe macht zufällige Aktivierungen sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Aktivierungsmatrix ersetzt hier einen CNN-Mini-Batch.
rng = np.random.default_rng(42)
activations = rng.normal(loc=3.0, scale=2.0, size=(8, 6))

# BatchNorm normalisiert jede Feature-Spalte im Mini-Batch.
feature_mean = activations.mean(axis=0, keepdims=True)
feature_std = activations.std(axis=0, keepdims=True)
normalized = (activations - feature_mean) / (feature_std + 1e-8)

# Dropout deaktiviert im Training zufällige Aktivierungen.
dropout_rate = 0.5
keep_mask = rng.random(normalized.shape) >= dropout_rate
train_output = normalized * keep_mask / (1.0 - dropout_rate)

# In der Auswertung bleibt Dropout ausgeschaltet.
eval_output = normalized.copy()

# Diese Kennzahlen zeigen die unterschiedlichen Effekte.
zero_share = np.mean(train_output == 0.0)
train_std = train_output.std()
eval_std = eval_output.std()

print("BatchNorm: Mittelwert nach Normalisierung =", round(float(normalized.mean()), 3))
print("BatchNorm: Standardabweichung danach =", round(float(normalized.std()), 3))
print("Dropout im Training: deaktivierter Anteil =", round(float(zero_share), 2))
print("Training-Std mit Dropout =", round(float(train_std), 3))
print("Auswertungs-Std ohne Dropout =", round(float(eval_std), 3))

# Die Grafik vergleicht dieselben Aktivierungen in beiden Modi.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train_output.ravel(), bins=12, alpha=0.65, label="Training mit Dropout")
ax.hist(eval_output.ravel(), bins=12, alpha=0.65, label="Auswertung ohne Dropout")
ax.set_title("Dropout verändert nur den Trainingsmodus")
ax.set_xlabel("Aktivierungswert nach BatchNorm")
ax.set_ylabel("Häufigkeit")
ax.legend()
plt.show()



### **1.3. Scheduler und Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_03.jpg?v=1784815826" width="250">



>* Scheduler steuern die Lernrate gezielt
>* Erst grob lernen, dann fein anpassen

>* Scheduler passend zur Trainingssituation wählen
>* Validierungsmetriken gegen Überanpassung prüfen

>* Early Stopping spart Zeit und verhindert Überanpassung
>* Geduldsspanne schützt vor zu frühem Abbruch



In [ ]:
#@title Python-Code - Scheduler und Stopping

# Dieses Beispiel zeigt Scheduler und Early Stopping.
# Wir simulieren Validierungsverluste über mehrere Epochen.
# Die Ausgabe zeigt Lernrate und Stoppzeitpunkt.

import numpy as np
import matplotlib.pyplot as plt

# Feste Werte machen den Trainingsverlauf reproduzierbar.
epochs = np.arange(1, 21)
validation_loss = np.array([
    0.92, 0.78, 0.66, 0.58, 0.52,
    0.49, 0.47, 0.465, 0.468, 0.466,
    0.469, 0.471, 0.474, 0.476, 0.479,
    0.481, 0.484, 0.486, 0.489, 0.492,
])

# Diese Parameter steuern Scheduler und Early Stopping.
learning_rate = 0.10
scheduler_patience = 2
stop_patience = 5
min_delta = 0.003

# Diese Listen speichern den Verlauf für die Grafik.
learning_rates = []
best_loss = float("inf")
best_epoch = 0
bad_epochs = 0
stop_epoch = epochs[-1]

# Jede Epoche prüft Verbesserung, Scheduler und Stoppregel.
for epoch, loss in zip(epochs, validation_loss):
    learning_rates.append(learning_rate)
    improved = loss < best_loss - min_delta

    if improved:
        best_loss = loss
        best_epoch = epoch
        bad_epochs = 0
    else:
        bad_epochs = bad_epochs + 1

    if bad_epochs > 0 and bad_epochs % scheduler_patience == 0:
        learning_rate = learning_rate * 0.5

    if bad_epochs >= stop_patience:
        stop_epoch = epoch
        break

# Nur der tatsächlich gelaufene Teil wird angezeigt.
shown_epochs = epochs[:len(learning_rates)]
shown_loss = validation_loss[:len(learning_rates)]
shown_rates = np.array(learning_rates)

# Kurze Textausgabe fasst die Entscheidung zusammen.
print("Beste Epoche:", int(best_epoch))
print("Bester Validierungsverlust:", round(float(best_loss), 3))
print("Early Stopping bei Epoche:", int(stop_epoch))
print("Letzte Lernrate:", round(float(shown_rates[-1]), 4))

# Die Grafik verbindet Validierungsverlust und Lernrate anschaulich.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(shown_epochs, shown_loss, marker="o", label="Validierungsverlust")
ax.plot(shown_epochs, shown_rates, marker="s", label="Lernrate")
ax.axvline(stop_epoch, color="red", linestyle="--", label="Stopp")
ax.set_title("Scheduler senkt die Lernrate, Early Stopping beendet Training")
ax.set_xlabel("Epoche")
ax.set_ylabel("Wert")
ax.legend()
plt.show()



## **2. Transfer Learning**

### **2.1. MobileNetV3 Small**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_01.jpg?v=1784815813" width="250">



>* Effizient für Geräte mit begrenzten Ressourcen
>* Vorwissen spart Training für neue Bildaufgaben

>* Vortrainiertes Modell liefert allgemeine Bildmerkmale
>* Einfrieren spart Daten, Zeit und Überanpassung

>* Vorwissen nutzen, nur Klassifikationskopf neu trainieren
>* Gut bei ähnlichen Bildern und knappen Ressourcen



### **2.2. Schichten gezielt einfrieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_02.jpg?v=1784815815" width="250">



>* Vortrainierte Schichten liefern übertragbare Bildmerkmale
>* Einfrieren spart Training und reduziert Überanpassung

>* Kleine, ähnliche Datensätze: Merkmalsextraktor einfrieren
>* Andere Domänen: höhere Schichten vorsichtig anpassen

>* Einfrieren strategisch prüfen und schrittweise anpassen
>* Leichte eingefrorene Modelle sparen Ressourcen



In [ ]:
#@title Python-Code - Schichten gezielt einfrieren

# Dieses Beispiel zeigt gezieltes Einfrieren von Schichten.
# Trainierbare Gewichte werden wie bei Transfer Learning gezählt.
# Die Ausgabe vergleicht drei einfache Einfrierstrategien.

import numpy as np

# Jede Schicht erhält eine kleine, erfundene Parameterzahl.
layer_names = np.array(["Kanten", "Texturen", "Formen", "Objekte", "Kopf"])
parameter_counts = np.array([1200, 3400, 6800, 9200, 900])

# Diese Strategien ahmen typische Transfer-Learning-Entscheidungen nach.
strategies = {
    "nur Kopf trainieren": np.array([False, False, False, False, True]),
    "letzten Block auftauen": np.array([False, False, False, True, True]),
    "alles trainieren": np.array([True, True, True, True, True]),
}

# Eine einfache Prüfung verhindert unpassende Maskenlängen.
for strategy_name, trainable_mask in strategies.items():
    if len(trainable_mask) != len(layer_names):
        raise ValueError("Jede Strategie braucht eine Entscheidung pro Schicht.")

# Die Gesamtzahl dient als Bezug für Prozentwerte.
total_parameters = int(parameter_counts.sum())
print("Strategie | trainierbare Schichten | trainierbare Parameter")

# Für jede Strategie zählen wir nur freigegebene Schichten.
for strategy_name, trainable_mask in strategies.items():
    trainable_layers = layer_names[trainable_mask]
    trainable_parameters = int(parameter_counts[trainable_mask].sum())
    trainable_percent = 100 * trainable_parameters / total_parameters
    layer_text = ", ".join(trainable_layers.tolist())
    print(f"{strategy_name} | {layer_text} | {trainable_parameters} ({trainable_percent:.1f}%)")

# Die letzte Zeile fasst die wichtigste Transfer-Learning-Idee zusammen.
print("Merke: Einfrieren spart Training, weil weniger Gewichte verändert werden.")



### **2.3. Klassifikationskopf trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_03.jpg?v=1784815817" width="250">



>* Eingefrorenes Modell liefert allgemeine Bildmerkmale
>* Neuer Kopf ordnet Merkmale Zielklassen zu

>* Schnelleres Training mit wenigen gelabelten Bildern
>* Kopf und Daten müssen zur Aufgabe passen

>* Validierungsleistung wichtiger als Trainingsgenauigkeit
>* Fehlerarten an Anwendungskontext bewerten



## **3. Transfer bewerten**

### **3.1. Inferenzzeit messen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_01.jpg?v=1784815819" width="250">



>* Inferenzzeit zeigt praktische Reaktionsgeschwindigkeit.
>* Immer auf Zielgerät und Einsatzkontext messen.

>* Einzelvorhersagen und Stapel getrennt messen
>* Mehrfach testen, Vorverarbeitung klar einbeziehen

>* Laufzeit immer mit Qualität und Größe abwägen
>* Einsatzkontext bestimmt das passende Transfermodell



In [ ]:
#@title Python-Code - Inferenzzeit messen

# Dieses Beispiel misst Inferenzzeit ohne externe Modelle.
# Zwei einfache Klassifikatoren zeigen praktische Laufzeitunterschiede.
# Die Ausgabe vergleicht Genauigkeit und Millisekunden pro Bild.

import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
X = digits.data

y = digits.target
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung trennt Training und spätere Inferenzmessung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)
if X_test_scaled.shape[0] < 100:
    raise ValueError("Zu wenige Testbilder für eine stabile Messung.")

# Ein sehr einfaches Modell dient als schneller Vergleichspunkt.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train_scaled, y_train)

# Logistische Regression ist genauer, aber rechenintensiver.
transfer_like_model = LogisticRegression(max_iter=300, random_state=42)
transfer_like_model.fit(X_train_scaled, y_train)

# Viele Wiederholungen machen die Messung weniger zufällig.
repeats = 40
batch_size = X_test_scaled.shape[0]

# NumPy liefert eine einfache CPU-Zeitmessung in Sekunden.
start_time = np.datetime64("now", "ns")
for _ in range(repeats):
    baseline_predictions = baseline_model.predict(X_test_scaled)
end_time = np.datetime64("now", "ns")

baseline_seconds = (end_time - start_time) / np.timedelta64(1, "s")
baseline_ms = 1000 * baseline_seconds / (repeats * batch_size)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

# Dieselbe Messmethode macht beide Modelle fair vergleichbar.
start_time = np.datetime64("now", "ns")
for _ in range(repeats):
    transfer_predictions = transfer_like_model.predict(X_test_scaled)
end_time = np.datetime64("now", "ns")

transfer_seconds = (end_time - start_time) / np.timedelta64(1, "s")
transfer_ms = 1000 * transfer_seconds / (repeats * batch_size)
transfer_accuracy = accuracy_score(y_test, transfer_predictions)

# Die Ergebnisse verbinden Qualität und praktische Nutzbarkeit.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testbilder pro Stapel: {batch_size}")
print(f"Dummy: Genauigkeit {baseline_accuracy:.3f}, {baseline_ms:.4f} ms/Bild")
print(f"LogReg: Genauigkeit {transfer_accuracy:.3f}, {transfer_ms:.4f} ms/Bild")
print("Merke: Schnellere Inferenz ist nur zusammen mit Genauigkeit sinnvoll.")



### **3.2. Genauigkeit vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_02.jpg?v=1784815821" width="250">



>* Genauigkeit hängt von Aufgabe und Daten ab
>* Immer auf separaten Testdaten prüfen

>* Gesamtgenauigkeit kann bei Ungleichgewicht täuschen
>* Bewerte Fehler nach Anwendungskontext

>* Gleiche Bedingungen für faire Modellvergleiche
>* Genauigkeit gegen Ressourcen und Einsatz abwägen



In [ ]:
#@title Python-Code - Genauigkeit vergleichen

# Wir vergleichen Genauigkeit unter fairen Bedingungen.
# Ein Klassifikator simuliert einen eingefrorenen Merkmalsextraktor.
# Die Ausgabe zeigt Gesamtgenauigkeit und Klassenfehler.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Der kleine Zifferndatensatz ist offline verfügbar.
digits = load_digits()
X = digits.data
y = digits.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Bilddaten und Zielwerte passen nicht zusammen.")

# Wir nutzen dieselbe Aufteilung für einen fairen Vergleich.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Das einfache Modell steht für einen kleinen Transfer-Kopf.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train_scaled, y_train)

# Genauigkeit wird nur auf ungesehenen Testdaten gemessen.
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

# Die Konfusionsmatrix zeigt, welche Klassen verwechselt werden.
labels = np.arange(10)
cm = confusion_matrix(y_test, y_pred, labels=labels)
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

# Eine kurze Tabelle macht die schwächsten Klassen sichtbar.
summary = pd.DataFrame(
    {"Ziffer": labels, "Klassengenauigkeit": per_class_accuracy}
)
weakest = summary.sort_values("Klassengenauigkeit").head(3)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit insgesamt: {accuracy:.3f}")
print("Schwächste Klassen nach Klassengenauigkeit:")
print(weakest.to_string(index=False, formatters={"Klassengenauigkeit": "{:.3f}".format}))



### **3.3. Transfer Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_03.jpg?v=1784815823" width="250">



>* Modelle fair unter gleichen Bedingungen vergleichen
>* Praxisnutzen statt nur Genauigkeit bewerten

>* Kleinen Datensatz mit stabiler Aufteilung wählen
>* Genauigkeit, Tempo, Größe und Offline-Fallback messen

>* Modellwahl passend zum Einsatz begründen
>* Kompromisse bei Genauigkeit, Tempo, Offlinebetrieb abwägen



In [ ]:
#@title Python-Code - Transfer Mini Projekt

# Dieses Mini Projekt vergleicht zwei Transfer Varianten.
# Simulierte Merkmale ersetzen hier heruntergeladene CNN Gewichte.
# Die Tabelle zeigt Genauigkeit, Tempo, Größe und Offlinefähigkeit.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Bildmerkmale wie aus eingefrorenen CNNs.
features, labels = make_classification(
    n_samples=900, n_features=20, n_informative=10, n_redundant=4,
    n_classes=3, class_sep=1.4, random_state=42
)

# Die Aufteilung bleibt für beide Varianten identisch.
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
test_scaled = scaler.transform(test_features)

# Zwei Köpfe stehen für zwei Transfer Learning Entscheidungen.
models = {
    "kleiner Kopf": LogisticRegression(C=0.4, max_iter=300, random_state=42),
    "größerer Kopf": LogisticRegression(C=4.0, max_iter=300, random_state=42),
}

# Wir sammeln praxisnahe Kennzahlen statt nur Genauigkeit.
rows = []
for name, model in models.items():
    model.fit(train_scaled, train_labels)
    predictions = model.predict(test_scaled)
    accuracy = accuracy_score(test_labels, predictions)
    parameter_count = model.coef_.size + model.intercept_.size
    size_kb = parameter_count * 8 / 1024
    speed_score = 1000 / (1 + parameter_count)
    offline_ready = name == "kleiner Kopf"
    rows.append([name, accuracy, speed_score, size_kb, offline_ready])

# Eine kompakte Tabelle macht die Zielkonflikte sichtbar.
columns = ["Modell", "Genauigkeit", "Tempo Punkte", "Größe KB", "Offline"]
results = pd.DataFrame(rows, columns=columns)
results["Genauigkeit"] = results["Genauigkeit"].round(3)
results["Tempo Punkte"] = results["Tempo Punkte"].round(1)
results["Größe KB"] = results["Größe KB"].round(2)

# Eine einfache Nutzwertzahl verbindet mehrere Entscheidungskriterien.
results["Nutzwert"] = (
    results["Genauigkeit"] * 70 + results["Tempo Punkte"] * 0.2
    - results["Größe KB"] * 2 + results["Offline"].astype(int) * 5
)

# Die Empfehlung folgt aus dem höchsten Nutzwert.
best_index = int(results["Nutzwert"].idxmax())
best_model = results.loc[best_index, "Modell"]

print("scikit-learn Version:", sklearn.__version__)
print(results.to_string(index=False))
print("Empfehlung:", best_model)

# Das Balkendiagramm zeigt den Vergleich auf einen Blick.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(results["Modell"], results["Nutzwert"], color=["seagreen", "steelblue"])
ax.set_title("Transfer Mini Projekt: Modellwahl nach Nutzwert")
ax.set_xlabel("Transfer Variante")
ax.set_ylabel("Nutzwert Punkte")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Transfer mit PyTorch**</font>


In this lecture, you learned to:
- Setzen Bildaugmentation, Dropout, Batch Normalization, Scheduler und Early Stopping kontrolliert ein. 
- Nutzen ein leichtes vortrainiertes torchvision-Modell als eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Modelle hinsichtlich Genauigkeit, Laufzeit, Größe und Offline-Fallback. 

In the next Module (Module 19), we will go over 'Sequenzen und Autoencoder'